# Session 2 · Campus food-ordering agent

**BGS College · V Naga Sai Rahul · 75-minute guided build**

Save your own copy. All cart, coupon, bill and tracking actions are **simulated**. No restaurant order or payment is made. The model makes real API requests with your eligible account; free access and quota vary.

| Minutes | Activity |
|---|---|
| 0–10 | Private setup and ordinary Python tools |
| 10–25 | Tool descriptions and a visible agent loop |
| 25–40 | Run an order and inspect the trace |
| 40–55 | Inspect the completed cart, coupon and bill |
| 55–65 | Use the chat interface and verify the bill |
| 65–75 | Optional tracking tool and saved evidence |

### If a model response is unavailable

Each model request waits up to 60 seconds. Temporary failures get one limited retry; every physical attempt counts against the same 40-request budget and five-second request spacing. Tools already executed are never replayed by that retry. A failed response stays pending, not a fake completed order. Later demo cells preserve the interrupted turn.

Already running? Refresh to load the saved source, then rerun the large helper containing `class FoodAgent` (code cell 5). It upgrades an existing `agent` without resetting its cart, trace or counter. Continue with `run_step("Resume pending order", agent.resume)` in a new cell, or run the chat-interface cell and use **Retry pending response**. Reset simulation is a separate deliberate choice that clears local order state but never refunds provider usage.


**Label the live model connection**

**What this cell does:** Set the access label used in the notebook evidence.

**How it works and how to run:** Run before the other cells. LIVE_API labels later model requests; it does not make the cart or payment real.

**What to expect:** No output is normal. This assignment makes no model or network request.


In [ ]:
# Later agent requests use the live provider; food actions remain simulated.
MODE = "LIVE_API"

## 1 · Setup and private key

Use the same private-key route as Session 1. The setup preserves Colab’s required authentication constraint and any already importable widget version. A CPU runtime is sufficient.

**Prepare the agent packages**

**What this cell does:** Install the SDK while preserving the widget version and active Colab authentication requirements.

**How it works and how to run:** Run once in a fresh runtime. The cell reads package metadata, selects compatible dependency arguments and runs pip in this notebook Python.

**What to expect:** A Setup ready line shows the installed versions. No model reply is requested.


In [ ]:
# Read installed package versions and the dependency requirements published by Colab.
import importlib.metadata as metadata
# Use this notebook Python to run pip, so packages install in the runtime we are using.
import subprocess, sys
# Parse package requirements so Colab authentication version rules can be preserved.
from packaging.requirements import Requirement

# Try the supported notebook feature first, then use the explicit fallback if it is unavailable.
try:
    # Check which widget version is already available before deciding what to install.
    import ipywidgets as native_widgets
    # Reuse an available widget version instead of replacing Colab widgets.
    widgets_pin = "ipywidgets==" + native_widgets.__version__
except ImportError:
    # Reuse the available widget version, or use the prepared fallback when widgets are missing.
    widgets_pin = "ipywidgets==8.1.9"
# Try the supported notebook feature first, then use the explicit fallback if it is unavailable.
try:
    # Respect Colab authentication requirements when installing the SDK.
    requirements = map(Requirement, metadata.requires("google-colab") or [])
    # Keep active google-auth requirements; outside Colab, an empty list adds no extra constraint.
    auth_constraints = [str(r) for r in requirements
        if r.name.lower().replace("_", "-") == "google-auth"
        and (r.marker is None or r.marker.evaluate())]
except metadata.PackageNotFoundError:
    # Keep active google-auth requirements; outside Colab, an empty list adds no extra constraint.
    auth_constraints = []
# Install the prepared versions in this notebook runtime.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "google-genai==2.12.0", widgets_pin, *auth_constraints])
# Show the versions actually installed, so setup can be checked before continuing.
print("Setup ready:", metadata.version("google-genai"), "SDK;",
      metadata.version("google-auth"), "google-auth;",
      metadata.version("ipywidgets"), "ipywidgets")

**Load the agent imports**

**What this cell does:** Load the SDK, JSON formatter, hidden key input and interactive notebook controls.

**How it works and how to run:** Run after package installation. MODEL selects the prepared provider route; model availability and account access must be verified before class.

**What to expect:** No output is normal. Imports and settings make no model request.


In [ ]:
# Load the Google Gen AI SDK used to create the model client.
from google import genai
# Load SDK message, configuration and schema types used by the API.
from google.genai import types
# Load hidden key input, readable JSON output and the notebook chat controls.
import getpass, json, ipywidgets as widgets
# Load the helper that renders the agent chat controls inside the notebook.
from IPython.display import display
# Use the prepared model setting consistently throughout the notebook.
MODEL = "gemini-3.5-flash-lite"

Create your own key in [Google AI Studio](https://aistudio.google.com/apikey). In Colab Secrets, add `GEMINI_API_KEY` and allow this notebook to access it. Never paste a key into a visible cell or chat message.

**Create the private client**

**What:** Read a private Gemini key and configure the client.

**Why:** Client construction is different from a successful model request.

**Steps:** Run after adding GEMINI_API_KEY in Secrets and granting notebook access.

**Expected:** Client ready without showing the key; timeout is 60 seconds and SDK retries are disabled.


In [ ]:
# Keep the credential in a temporary runtime variable; never print or hard-code its value.
api_key = None
# Try the supported notebook feature first, then use the explicit fallback if it is unavailable.
try:
    # Load Colab Secrets access so the key does not need to appear in notebook source.
    from google.colab import userdata
    # Try the supported notebook feature first, then use the explicit fallback if it is unavailable.
    try:
        # Read the private secret without displaying it.
        api_key = userdata.get("GEMINI_API_KEY")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        # Continue when this optional notebook feature or private secret is unavailable; the fallback below can handle setup.
        pass
except ImportError:
    # Continue when this optional notebook feature or private secret is unavailable; the fallback below can handle setup.
    pass
# Use the fallback or stop with a clear instruction when no nonempty credential is available.
if not api_key:
    # Prompt invisibly only when no accessible secret was found.
    api_key = getpass.getpass("Your Gemini API key (hidden): ").strip()
# Use the fallback or stop with a clear instruction when no nonempty credential is available.
if not api_key:
    # Stop setup and ask for a private credential instead of continuing with an empty key.
    raise RuntimeError("Add your own GEMINI_API_KEY in Colab Secrets, then rerun.")
# Configure one bounded client; constructing it does not generate a reply.
client = genai.Client(api_key=api_key, http_options=types.HttpOptions(
    timeout=60000, retry_options=types.HttpRetryOptions(attempts=1)))
# Remove this temporary variable after handing it to the client.
del api_key
# Confirm client setup without revealing the key; later generation cells use your account.
print("Client ready. The next cell that calls Gemini uses your account.")

## 2 · The tools and explicit loop

Run the provided helper unchanged. It validates each tool name and argument, records observations, and caps both model steps and tool calls. Model requests start at least five seconds apart, including after Reset simulation, so give each reply time to finish. A quota error stops the run; temporary transport/server failures get one bounded retry; quota and permission errors are not automatically retried. Automatic SDK function execution is disabled, so our code remains responsible for dispatch. Expand the helper if you want to inspect it.

**Define bounded tools, requests and resume**

**What:** Define tool contracts, local state, the explicit loop and readable notebook statuses.

**Why:** A failed model response must not repeat a tool action that already changed the cart.

**Steps:** Run the helper. It upgrades an existing agent in place, preserving cart, trace, counters and request pacing. Each transient request gets at most one explicit retry.

**Expected:** Definitions only; no model request. An existing interrupted turn can use agent.resume() or Retry pending response.


In [ ]:
"""Provided food-agent code. All ordering actions are a local simulation."""
# Load deep copying so returned menu data and trace records cannot edit the original objects.
import copy
# Load JSON formatting so the tool trace is easy to inspect.
import json
# Load a monotonic clock and sleep function for spacing model requests.
import time
# Recognise temporary network failures without exposing raw request details.
import httpx
# Load SDK message, configuration and schema types used by the API.
from google.genai import types

# Menu data is the source of prices used by the local bill tool.

# Each physical model request is bounded; SDK retries stay disabled.
REQUEST_TIMEOUT_MS = 60000
MAX_REQUEST_ATTEMPTS = 2
RETRY_DELAY_SECONDS = 2.0
TRANSIENT_STATUS_CODES = {408, 500, 502, 503, 504}
LAB_STEP_STATUS = {}


class LabRequestError(RuntimeError):
    """Safe classroom status: never contains the raw request or provider body."""
    def __init__(self, stage, message, status_code=None, retryable=False):
        super().__init__(message)
        self.stage, self.status_code, self.retryable = stage, status_code, retryable


def run_step(label, operation):
    # None means pending, not an invented reply or a successful order check.
    try:
        value = operation()
    except LabRequestError as error:
        LAB_STEP_STATUS[label] = {"status":"pending", "stage":error.stage, "code":error.status_code}
        print(f"{label}: {error}")
        return None
    except (RuntimeError, ValueError) as error:
        LAB_STEP_STATUS[label] = {"status":"pending", "type":type(error).__name__}
        print(f"{label}: {error}")
        return None
    except Exception as error:
        LAB_STEP_STATUS[label] = {"status":"pending", "type":type(error).__name__}
        print(f"{label}: the step could not finish ({type(error).__name__}). Ask the instructor to inspect the inputs.")
        return None
    LAB_STEP_STATUS[label] = {"status":"pending" if value is None else "complete"}
    return value

MENU = {
    "dosa": {"price": 40, "vegetarian": True},
    "idli": {"price": 30, "vegetarian": True},
    "lemon_rice": {"price": 50, "vegetarian": True},
    "tea": {"price": 20, "vegetarian": True},
}
# Bound work so a repeated tool cycle cannot run forever.
MAX_STEPS, MAX_TOOLS_PER_TURN, MAX_REQUESTS = 8, 12, 40
# Space model-request starts by at least the supplied interval; this does not guarantee provider quota.
MIN_REQUEST_INTERVAL_SECONDS = 5.0
# Select the prepared model ID; availability and account access must be verified before class.
MODEL = "gemini-3.5-flash-lite"
# Set the instructions that guide the model role and behavior on later requests.
SYSTEM = """You help with a SIMULATED campus food order. Use the available tools
for menu facts, cart changes, coupons and bills. Menu prices and tool results are
authoritative. Never claim an action succeeded unless its tool result confirms it.
STUDENT10 gives 10 percent off a subtotal of at least 100 INR. Ask when the item or
quantity is ambiguous. Never invent an item, price, order submission or payment.
You may update this local cart as requested. All orders are simulations."""


# Keep the simulated cart and coupon in Python state that can be checked independently of model text.
class OrderState:
    # Initialize the separate state and controls for each new instance.
    def __init__(self, fix_quantity=None):
        # Accept the old optional constructor argument for saved notebook compatibility.
        # All instances now use the validated requested quantity.
        self.fix_quantity = True
        # Start with an empty item-to-quantity dictionary for this simulation.
        self.cart = {}
        # Start or reset this cart with no coupon applied.
        self.coupon = False

    # Define how to clear the local cart and remove any accepted coupon for a fresh simulation.
    def reset(self):
        # Empty the stored item quantities for a fresh simulated order.
        self.cart.clear()
        # Start or reset this cart with no coupon applied.
        self.coupon = False

    # Calculate the bill from stored cart quantities and menu prices.
    def bill(self):
        # Calculate the subtotal from stored quantities and the menu prices, using Python arithmetic.
        subtotal = sum(MENU[item]["price"] * qty for item, qty in self.cart.items())
        # Apply integer 10 percent discount only when a coupon is active and the subtotal is at least 100 INR.
        discount = subtotal // 10 if self.coupon and subtotal >= 100 else 0
        # Return cart, subtotal, discount and total as checkable data, with an explicit simulation label.
        return {"cart": dict(self.cart), "subtotal_inr": subtotal,
                "discount_inr": discount, "total_inr": subtotal - discount,
                "simulated": True}

    # Dispatch only known tools after validating their argument contract.
    def execute(self, name, arguments, tracking=False):
        # Reject non-object arguments before reading fields or changing cart state.
        if not isinstance(arguments, dict):
            # Return a rejected-operation record before any forbidden state change can happen.
            return {"ok": False, "error": "ARGUMENT_OBJECT_REQUIRED"}
        # Copy arguments to a plain dictionary so validation and dispatch use a predictable structure.
        args = dict(arguments)
        # List the exact argument names each allowed tool accepts; empty sets mean no arguments.
        contracts = {"get_menu": set(), "get_cart": set(), "get_bill": set(),
                     "add_item": {"item_id", "quantity"}, "apply_coupon": {"code"}}
        # Expose the optional status capability only when the extension has been enabled.
        if tracking:
            # Add the optional read-only tracking contract only when that extension is enabled.
            contracts["get_tracking"] = set()
        # Reject a tool name that is not in the local allowlist.
        if name not in contracts:
            # Return a rejected-operation record before any forbidden state change can happen.
            return {"ok": False, "error": "UNKNOWN_TOOL"}
        # Reject extra or missing arguments before changing the cart.
        if set(args) != contracts[name]:
            # Return a rejected-operation record before any forbidden state change can happen.
            return {"ok": False, "error": "UNEXPECTED_OR_MISSING_ARGUMENT"}
        # For a menu request, return a separate copy of the allowed items and prices.
        if name == "get_menu":
            # Return a successful tool observation based on the current simulated state.
            return {"ok": True, "menu": copy.deepcopy(MENU)}
        # For a cart request, expose the current stored quantities without changing them.
        if name == "get_cart":
            # Return a successful tool observation based on the current simulated state.
            return {"ok": True, "cart": dict(self.cart)}
        # For a bill request, calculate from the actual Python cart state.
        if name == "get_bill":
            # Return a successful tool observation based on the current simulated state.
            return {"ok": True, **self.bill()}
        # Return an explicit simulated status, with no claim that an order was submitted.
        if name == "get_tracking":
            # Return a successful tool observation based on the current simulated state.
            return {"ok": True, "status": "Simulation only; no real order was submitted."}
        # Validate the add-item arguments before applying a local cart change.
        if name == "add_item":
            # Read the requested menu item and quantity only after the argument names have been checked.
            item, qty = args["item_id"], args["quantity"]
            # Reject an unknown item or a non-text item ID so the model cannot invent menu entries.
            if not isinstance(item, str) or item not in MENU:
                # Return a rejected-operation record before any forbidden state change can happen.
                return {"ok": False, "error": "UNKNOWN_MENU_ITEM"}
            # Reject booleans as well as unsupported numeric quantities.
            if type(qty) is not int or not 1 <= qty <= 5:
                # Return a rejected-operation record before any forbidden state change can happen.
                return {"ok": False, "error": "QUANTITY_MUST_BE_INTEGER_1_TO_5"}
            # Use the requested quantity after all input checks have passed.
            added = qty
            # Reject a cart change that would exceed the per-item stored quantity limit.
            if self.cart.get(item, 0) + added > 20:
                # Return a rejected-operation record before any forbidden state change can happen.
                return {"ok": False, "error": "CART_ITEM_LIMIT"}
            # Store the new quantity only after validating the cart limit.
            self.cart[item] = self.cart.get(item, 0) + added
            # Return a successful tool observation based on the current simulated state.
            return {"ok": True, "cart": dict(self.cart)}
        # Read the coupon code after its argument shape has already been validated.
        code = args["code"]
        # Reject coupon values other than the exact fictional STUDENT10 code.
        if not isinstance(code, str) or code != "STUDENT10":
            # Return a rejected-operation record before any forbidden state change can happen.
            return {"ok": False, "error": "UNKNOWN_COUPON"}
        # Enforce coupon eligibility against actual cart state.
        if self.bill()["subtotal_inr"] < 100:
            # Return a rejected-operation record before any forbidden state change can happen.
            return {"ok": False, "error": "MINIMUM_SUBTOTAL_100_NOT_MET"}
        # Record the accepted coupon only after its code and subtotal eligibility have passed.
        self.coupon = True
        # Return a successful tool observation based on the current simulated state.
        return {"ok": True, **self.bill()}


# Describe callable operations; declarations themselves execute nothing.
def tool_declarations(tracking=False):
    # Use the SDK object type when describing tool argument schemas.
    object_type = types.Type.OBJECT
    # Describe each allowed tool name, purpose, argument types and required fields to the model.
    specs = [
        ("get_menu", "Read allowed item IDs, prices in INR and vegetarian flags.", {}, []),
        ("get_cart", "Read the currently stored item quantities.", {}, []),
        ("add_item", "Add quantity 1 to 5 of a known menu item to this simulated cart.",
         {"item_id": types.Schema(type=types.Type.STRING),
          "quantity": types.Schema(type=types.Type.INTEGER)}, ["item_id", "quantity"]),
        ("apply_coupon", "Apply STUDENT10 only when subtotal is at least 100 INR.",
         {"code": types.Schema(type=types.Type.STRING)}, ["code"]),
        ("get_bill", "Calculate and return the current cart subtotal, discount and total.", {}, []),
    ]
    # Expose the optional status capability only when the extension has been enabled.
    if tracking:
        # Add a description for the optional simulation status tool.
        specs.append(("get_tracking", "Read the simulated order status; no real order exists.", {}, []))
    # Convert the allowed tool descriptions and argument schemas into the SDK declaration format.
    return [types.Tool(function_declarations=[types.FunctionDeclaration(
        name=name, description=description,
        parameters=types.Schema(type=object_type, properties=properties, required=required))
        for name, description, properties, required in specs])]


# Keep model conversation, local tool state and observable trace separate.
class FoodAgent:
    def __init__(self, client, state, model=MODEL, tracking=False):
        self.client, self.state, self.model = client, state, model
        self.tracking = tracking
        self.messages, self.trace = [], []
        self.requests_used = 0
        self._request_clock = time.monotonic
        self._request_sleeper = time.sleep
        self._last_request_started_at = None
        self._pending_turn = None
        self.last_result, self.last_error = None, None

    @property
    def has_pending_turn(self):
        return self._pending_turn is not None

    @property
    def pending_question(self):
        return self._pending_turn.get("question") if self._pending_turn else None

    @property
    def can_resume(self):
        pending = self._pending_turn
        return bool(pending is not None and not pending.get("reset_only", False)
                    and pending["step"] < MAX_STEPS and self.requests_used < MAX_REQUESTS)

    def reset(self):
        # This is an explicit new simulation, not a refund of provider requests.
        self.state.reset()
        self.messages.clear()
        self.trace.clear()
        self._pending_turn = None
        self.last_result, self.last_error = None, None

    def _wait_for_request_start(self):
        # Preserve the original five-second pacing, including retries and resume.
        while self._last_request_started_at is not None:
            remaining = (self._last_request_started_at + MIN_REQUEST_INTERVAL_SECONDS
                         - self._request_clock())
            if remaining <= 0:
                break
            self._request_sleeper(remaining)
        self._last_request_started_at = self._request_clock()

    def _request(self, config):
        # Freeze this one decision request; do not replay any executed tools.
        payload = copy.deepcopy(self.messages)
        for attempt in range(1, MAX_REQUEST_ATTEMPTS + 1):
            if self.requests_used >= MAX_REQUESTS:
                raise RuntimeError("Notebook request limit reached; stop and review the trace.")
            self._wait_for_request_start()
            self.requests_used += 1
            try:
                reply = self.client.models.generate_content(
                    model=self.model, contents=copy.deepcopy(payload), config=config)
            except Exception as error:
                raw_code = getattr(error, "code", None)
                code = raw_code if isinstance(raw_code, int) else None
                retryable = code in TRANSIENT_STATUS_CODES or isinstance(
                    error, (TimeoutError, httpx.TimeoutException, httpx.ConnectError))
                self.last_error = {"stage":"generation", "type":type(error).__name__,
                                   "code":code, "attempt":attempt}
                if retryable and attempt < MAX_REQUEST_ATTEMPTS and self.requests_used < MAX_REQUESTS:
                    print("Temporary model-service issue; retrying this response once. Existing cart actions will not be repeated.")
                    self._request_sleeper(RETRY_DELAY_SECONDS)
                    continue
                if self.requests_used >= MAX_REQUESTS:
                    message = "Notebook request limit reached. Your cart and trace are kept; stop and review them."
                elif code == 429:
                    message = "Quota/rate limit reached. Stop requests and check your account with the instructor. Your cart and trace are kept."
                elif code in (401,403):
                    message = "Check your private key and account permissions with the instructor. Your current simulation is kept."
                elif code == 404:
                    message = "The configured model is unavailable to this account. Verify its setting with the instructor."
                elif code == 400:
                    message = "The provider rejected the request. Ask the instructor to review the model and tool settings."
                elif retryable:
                    message = "The model service is still unavailable after the limited retry. The order is pending; its cart and trace are kept."
                else:
                    message = "The model response could not complete. Check setup and connection with the instructor; no completed order is claimed."
                raise LabRequestError("generation", message, code, retryable) from None
            self.last_error = None
            return reply

    def run(self, question):
        self.last_result = None
        if self.has_pending_turn:
            raise RuntimeError("A previous turn is pending. Use Retry pending response / agent.resume(), or explicitly Reset simulation before a new order.")
        if not isinstance(question, str) or not question.strip() or len(question) > 1200:
            raise ValueError("Use one short order or question, up to 1200 characters.")
        self.messages.append(types.Content(role="user", parts=[types.Part.from_text(text=question)]))
        self._pending_turn = {"question":question, "step":0, "tools_used":0, "reset_only":False}
        return self._continue_turn()

    def resume(self):
        # Continue after the last observation, without adding the order a second time.
        self.last_result = None
        if not self.has_pending_turn:
            raise ValueError("There is no pending response to resume.")
        if not self.can_resume:
            raise RuntimeError("This pending turn cannot resume within its limits or has an incomplete tool batch. Inspect the trace before an explicit Reset simulation; the request budget is unchanged.")
        return self._continue_turn()

    def _continue_turn(self):
        pending = self._pending_turn
        for step in range(pending["step"], MAX_STEPS):
            config = types.GenerateContentConfig(
                http_options=types.HttpOptions(timeout=REQUEST_TIMEOUT_MS,
                    retry_options=types.HttpRetryOptions(attempts=1)),
                system_instruction=SYSTEM, tools=tool_declarations(self.tracking),
                max_output_tokens=1024,
                automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
            reply = self._request(config)
            # A returned model decision consumes a logical step, also across resume.
            pending["step"] = step + 1
            if not reply.candidates or "MAX_TOKENS" in str(reply.candidates[0].finish_reason):
                raise RuntimeError("No complete model result; no tool from this response was executed. The turn remains pending.")
            content = reply.candidates[0].content
            if content is None:
                raise RuntimeError("Model returned no content. The turn remains pending.")
            calls = [part.function_call for part in content.parts or [] if part.function_call]
            if not calls:
                answer = "".join(p.text or "" for p in content.parts or [] if not p.thought)
                if not answer.strip():
                    raise RuntimeError("No final answer; inspect the trace. The turn remains pending.")
                self.messages.append(content)
                self.last_result = answer
                self._pending_turn = None
                return answer
            # Reject an oversized batch before adding unmatched calls to history.
            if pending["tools_used"] + len(calls) > MAX_TOOLS_PER_TURN:
                pending["reset_only"] = True
                raise RuntimeError("Tool-call limit reached before executing this batch; task incomplete. Inspect the recorded trace.")
            self.messages.append(content)
            observations = []
            try:
                for call in calls:
                    pending["tools_used"] += 1
                    args = dict(call.args or {})
                    result = self.state.execute(call.name, args, self.tracking)
                    self.trace.append({"step":step+1, "tool":call.name,
                        "arguments":copy.deepcopy(args), "result":copy.deepcopy(result)})
                    data = {"name":call.name, "response":result}
                    if getattr(call,"id",None):
                        data["id"] = call.id
                    observations.append(types.Part(function_response=types.FunctionResponse(**data)))
            except Exception as error:
                # Some earlier calls may already have changed the cart: never replay them.
                pending["reset_only"] = True
                self.last_error = {"stage":"tool execution", "type":type(error).__name__}
                raise RuntimeError("A tool batch could not finish. Partial cart and trace are kept. Inspect them, then explicitly Reset simulation; this batch will not be replayed.") from None
            self.messages.append(types.Content(role="user", parts=observations))
        pending["reset_only"] = True
        raise RuntimeError("Step limit reached; task incomplete. Inspect recorded observations before an explicit reset.")

    def show_trace(self):
        print(json.dumps(self.trace, indent=2))

def _infer_pending_turn(messages):
    # Old helpers did not store a pending marker. Recover it only from visible protocol history.
    if not messages:
        return None
    start = None
    for index in range(len(messages)-1, -1, -1):
        message = messages[index]
        if message.role == "user" and any(p.text for p in message.parts or []):
            start = index
            break
    if start is None:
        return {"question":"", "step":MAX_STEPS, "tools_used":0, "reset_only":True}
    turn = messages[start:]
    question = "".join(p.text or "" for p in turn[0].parts or [])
    last = turn[-1]
    if (last.role == "model" and not any(p.function_call for p in last.parts or [])
            and any((p.text or "").strip() for p in last.parts or [] if not p.thought)):
        return None
    steps = sum(m.role == "model" for m in turn)
    tool_count = sum(bool(p.function_response) for m in turn for p in m.parts or [])
    # Unanswered model calls may represent an interrupted, partially executed batch.
    reset_only = last.role == "model" or steps >= MAX_STEPS or tool_count > MAX_TOOLS_PER_TURN
    return {"question":question, "step":steps, "tools_used":tool_count, "reset_only":reset_only}


def upgrade_agent(existing):
    # Preserve the actual cart, trace, messages, attempt counter and pacing hooks.
    if existing is None or type(existing).__name__ != "FoodAgent":
        raise ValueError("Use an existing FoodAgent instance.")
    pending = getattr(existing,"_pending_turn",None)
    if not hasattr(existing,"_pending_turn"):
        pending = _infer_pending_turn(existing.messages)
    # Refresh state methods too, while preserving every stored cart quantity and coupon.
    # Existing completed actions are evidence; this update does not replay or rewrite them.
    if type(existing.state).__name__ == "OrderState":
        existing.state.__class__ = OrderState
        existing.state.fix_quantity = True
    existing.__class__ = FoodAgent
    existing._pending_turn = pending
    existing.last_result = None
    existing.last_error = None
    return existing


# Re-running the helper upgrades the open notebook without silently clearing its order.
if globals().get("agent") is not None and type(globals()["agent"]).__name__ == "FoodAgent":
    upgrade_agent(globals()["agent"])
    print("Agent updated; the existing cart, trace, request count and pacing are kept.")


**Start or retain the simulation**

**What this cell does:** Create a correct empty cart and agent only when no existing agent is present.

**How to run:** Run after the helper. Re-running this cell reuses the upgraded agent, preserving its cart, request counter and any pending turn.

**What to expect:** Student and Instructor versions both honor the requested quantity from the start. No model request.


In [ ]:
# Reuse an existing simulation so rerunning setup never erases a pending order.
existing_agent = globals().get("agent")
if existing_agent is not None and isinstance(existing_agent, FoodAgent):
    agent = existing_agent
    state = agent.state
    print("Existing simulation retained, including its request counter and any pending turn.")
else:
    # Every new cart uses the validated quantity requested by the user.
    state = OrderState()
    agent = FoodAgent(client, state, model=MODEL)
# Evidence becomes available only after a completed order has been inspected.
order_check = None
checked_bill = None
print("MODE:", MODE, "— actions are local simulations.")


### Run the tools before using the model

Call the Python tools directly: read the menu, add two dosas and inspect the cart. A quantity of 2 stores exactly two dosas, so the subtotal is 80 INR. The validation example rejects an unsupported quantity without changing the cart. These local operations make no model request.


**Inspect direct Python tool behavior**

**What this cell does:** Read the menu, add two dosas and show a local bill without a model.

**How to explain:** The cart contains two dosas and the subtotal is 80 INR. The validation example rejects `True` as a quantity and keeps the cart unchanged.

**What to expect:** A fresh cart is left ready for the next demonstration. If a model turn is pending, the entire demo pauses to preserve it.


In [ ]:
# Leave an interrupted model turn intact before demonstrating separate Python tools.
if agent.has_pending_turn:
    print("Direct tool demo paused: finish the pending response, or explicitly Reset simulation first.")
else:
    # Start a small local example; reset does not refund model requests.
    agent.reset()
    order_check, checked_bill = None, None
    print("Menu:", state.execute("get_menu", {}))
    # A quantity of 2 stores exactly two dosas in the cart.
    print("Add two dosas:", state.execute("add_item", {"item_id":"dosa", "quantity":2}))
    print("Local bill:", state.bill())
    # An invalid quantity is rejected without changing the stored order.
    before = dict(state.cart)
    print("Validation example:", state.execute("add_item", {"item_id":"dosa", "quantity":True}))
    assert state.cart == before
    # Leave an empty simulation ready for the model demonstration.
    agent.reset()


## 3 · Let the model select a tool

Ask for the menu first. Read the trace: tool name, arguments and returned observation. A sentence saying something happened is not sufficient evidence.

**Ask for the menu**

**What:** Let a real model choose the menu tool and inspect its returned observation.

**Why:** A claimed tool action needs actual trace evidence.

**Steps:** Temporary request failures get one retry. Re-running this same pending question resumes it; another pending task is preserved.

**Expected:** An actual model reply and trace, or pending status. Quota/access errors do not receive automatic retries.


In [ ]:
# Resume this exact menu question only; a different pending order is left intact.
MENU_QUESTION = "Show the vegetarian menu with prices. Use the menu tool."
menu_reply, order_check, checked_bill = None, None, None
LAB_STEP_STATUS.pop("Inspect order and bill", None)
if agent.has_pending_turn:
    if agent.pending_question == MENU_QUESTION and agent.can_resume:
        menu_reply = run_step("Menu response", agent.resume)
    else:
        print("Menu demo pending: another turn needs review or an explicit reset.")
else:
    agent.reset()
    menu_reply = run_step("Menu response", lambda: agent.run(MENU_QUESTION))
if menu_reply is not None:
    print(menu_reply)
agent.show_trace()


## 4 · Place a complete simulated order

Ask for two dosas and one tea, apply STUDENT10 if eligible, and get the bill. Follow the trace from the user request to the tool arguments and returned results. The tool stores the requested quantities from the start. A completed model sentence and the actual Python cart should agree.


**Place a complete order with tools**

**What this cell does:** Send one order for two dosas, one tea, an eligible coupon and a bill.

**How to explain:** Compare the model's tool arguments with the cart and bill returned by Python. The model chooses operations; the tools validate and execute them.

**What to expect:** An actual completed reply and tool trace, or a clear pending status. Re-running the exact pending order resumes its response without adding the food again.


In [ ]:
# Ask for one complete simulated order; its quantities and prices come from tools.
ORDER = "Add two dosas and one tea. Apply STUDENT10 if eligible, then show the bill. Use tools for every action and price."
order_reply, order_check, checked_bill, requested_cart_check = None, None, None, None
LAB_STEP_STATUS.pop("Inspect order and bill", None)
if agent.has_pending_turn:
    if agent.pending_question == ORDER and agent.can_resume:
        # Resume the response after the last tool result; never add the order twice.
        order_reply = run_step("Order response", agent.resume)
    else:
        print("Order demo pending: finish or review the earlier turn before starting a different order.")
else:
    agent.reset()
    order_reply = run_step("Order response", lambda: agent.run(ORDER))
if order_reply is not None:
    print(order_reply)
    requested_cart_check = state.cart == {"dosa":2,"tea":1}
agent.show_trace()
print("Current simulated bill (may be partial while pending):", state.bill())
print("Requested cart matches:", requested_cart_check)


### Inspect the completed cart and bill

This cell only reads and checks the order above. It does not call the model again, reset the cart or add more items. Two dosas at 40 INR and one tea at 20 INR give a subtotal of 100 INR. STUDENT10 discounts 10 INR, leaving a total of 90 INR. If the previous response is still pending, resume it before checking.


**Inspect the completed order and bill**

**What this cell does:** Read the completed cart and compare the full bill with the expected result.

**How to explain:** Two dosas and one tea cost 100 INR before STUDENT10; the discount is 10 INR and the total is 90 INR.

**What to expect:** `order_check` is True only when all bill fields match, False for a completed mismatch, or None when inspection is not ready. This cell makes no model request and never resets or changes the cart.


In [ ]:
# Read the completed order without another model request or any cart change.
order_check, checked_bill = None, None
LAB_STEP_STATUS.pop("Inspect order and bill", None)
if agent.has_pending_turn:
    print("Bill inspection pending: finish the previous response first. Its cart and trace are kept.")
else:
    # Find the last actual user request, ignoring the tool-response messages.
    last_question = next(("".join(part.text or "" for part in msg.parts or [])
        for msg in reversed(agent.messages)
        if msg.role == "user" and any(part.text for part in msg.parts or [])), None)
    if not agent.last_result or last_question != globals().get("ORDER"):
        print("Run the complete two-dosas-and-tea order above, then inspect its bill here.")
    else:
        checked_bill = state.bill()
        # Python checks the actual stored cart and arithmetic, independently of model prose.
        expected_bill = {"cart":{"dosa":2,"tea":1},"subtotal_inr":100,
            "discount_inr":10,"total_inr":90,"simulated":True}
        order_check = checked_bill == expected_bill
        LAB_STEP_STATUS["Inspect order and bill"] = {"status":"complete", "matches_expected":order_check}
        print("Expected: 2 dosas x 40 + 1 tea x 20 = 100 INR; discount 10 INR; total 90 INR.")
        print("Actual completed bill:", checked_bill)
        print("Checked order and bill:", order_check)
        if not order_check:
            print("The completed tool results differ from this order. Read the trace with the instructor.")


## 5 · A notebook chat interface

The interface uses the same bounded loop and local state. Try a follow-up such as “show my cart”. Use **Reset simulation** before starting a new order. It does not open a public server.

**Use notebook chat and deliberate resume**

**What:** Display Send, Retry pending response and Reset simulation around the same agent.

**Why:** Continuing an interrupted response is different from sending the original order again.

**Steps:** Send once. If unavailable, review the status and use Retry pending response when appropriate; reset is a separate explicit choice.

**Expected:** Readable status without a red request traceback; retries retain state and never silently replay executed tools.


In [ ]:
# Existing state is reused; a retry continues the response rather than replaying tools.
message = widgets.Text(placeholder="Ask about this simulated order", layout=widgets.Layout(width="70%"))
send = widgets.Button(description="Send")
retry_response = widgets.Button(description="Retry pending response", layout=widgets.Layout(width="210px"))
reset = widgets.Button(description="Reset simulation", layout=widgets.Layout(width="160px"))
log = widgets.Output()

def update_controls():
    retry_response.disabled = not agent.can_resume

def on_send(_):
    global order_check, checked_bill
    prompt = message.value.strip()
    if not prompt:
        return
    if agent.has_pending_turn:
        with log:
            print("An earlier turn is pending. Use Retry pending response, or inspect the trace before an explicit Reset simulation.")
        update_controls()
        return
    order_check, checked_bill = None, None
    LAB_STEP_STATUS.pop("Inspect order and bill", None)
    send.disabled = True
    try:
        with log:
            print("You:", prompt)
            reply = run_step("Chat response", lambda: agent.run(prompt))
            if reply is not None:
                print("Agent:", reply, "\n")
                message.value = ""
            else:
                print("Response pending. Current cart and trace are kept; no completed order is claimed.")
    finally:
        send.disabled = False
        update_controls()

def on_retry(_):
    global order_check, checked_bill
    order_check, checked_bill = None, None
    LAB_STEP_STATUS.pop("Inspect order and bill", None)
    send.disabled = True
    retry_response.disabled = True
    try:
        with log:
            reply = run_step("Resume pending response", agent.resume)
            if reply is not None:
                print("Agent:", reply, "\n")
                message.value = ""
    finally:
        send.disabled = False
        update_controls()

def on_reset(_):
    global order_check, checked_bill
    agent.reset()
    order_check, checked_bill = None, None
    LAB_STEP_STATUS.clear()
    log.clear_output()
    message.value = ""
    update_controls()

send.on_click(on_send)
retry_response.on_click(on_retry)
reset.on_click(on_reset)
update_controls()
display(widgets.VBox([widgets.HTML("<h3>Simulated campus food agent</h3>"),
    widgets.HBox([message,send]), widgets.HBox([retry_response,reset]),log]))


## 6 · Optional tool extension

The provided tracking function reports only that this is a simulation. Enable the tool, inspect its declaration, then ask for status. A model cannot submit a real order because no such action is available.

**Optionally inspect simulated tracking**

**What:** Expose the existing read-only tracking capability after the core exercise.

**Why:** Optional work must not erase or bypass a pending order.

**Steps:** Leave the flag False for the core lab. Enabled tracking pauses when another turn is pending.

**Expected:** No request when disabled; a real tracking reply or a truthful pending status when enabled.


In [ ]:
# Optional tracking never submits a real order and stays off by default.
ENABLE_TRACKING = False
tracking_reply = None
TRACKING_QUESTION = "Check the simulated order status using the tracking tool."
if not ENABLE_TRACKING:
    print("Optional tracking is disabled.")
elif agent.has_pending_turn:
    if agent.pending_question == TRACKING_QUESTION and agent.can_resume:
        tracking_reply = run_step("Tracking response", agent.resume)
    else:
        print("Tracking is pending: complete or review the earlier turn before enabling another action.")
else:
    agent.tracking = True
    tracking_reply = run_step("Tracking response", lambda: agent.run(TRACKING_QUESTION))
if tracking_reply is not None:
    print(tracking_reply)


## 7 · Save evidence

Keep one tool trace and one checked bill. Explain how the model selected tools, how Python validated arguments, and how the cart produced the final total. The `order_check` field records the completed bill inspection; `null` means it has not completed. Clear personal outputs before sharing. Return to the quiz after the lab.


**Save completed and pending evidence**

**What this cell does:** Summarise the bill inspection, pending state and request usage.

**How to explain:** The saved `checked_bill_snapshot` comes from the completed inspection. `null` means the check has not completed and is not a pass.

**What to expect:** A readable summary with `order_check`, even when a model response is unavailable. No additional model request.


In [ ]:
# Report only the completed bill inspection; a pending response is not a passed check.
active_agent = globals().get("agent")
pending = bool(active_agent is not None and (not hasattr(active_agent,"has_pending_turn") or active_agent.has_pending_turn))
evidence = {
    "mode": globals().get("MODE", "not configured"),
    "simulated": True,
    "order_check": None if pending else globals().get("order_check"),
    "checked_bill_snapshot": None if pending or globals().get("order_check") is None else globals().get("checked_bill"),
    "model_requests_used": getattr(active_agent,"requests_used",0),
    "pending_turn": pending,
    "can_resume": bool(getattr(active_agent,"can_resume",False)),
    "last_request_issue": getattr(active_agent,"last_error",None),
    "step_status": dict(globals().get("LAB_STEP_STATUS",{})),
}
print(json.dumps(evidence,indent=2))
print("null / pending means the check has not completed; it is not a pass.")


## Sources

- [Gemini function calling](https://ai.google.dev/gemini-api/docs/function-calling)
- [Google Gen AI Python SDK](https://github.com/googleapis/python-genai)
- [Microsoft AI agents lesson](https://github.com/microsoft/generative-ai-for-beginners/tree/main/17-ai-agents)